# 🔬 TRICARE Retriever 실험 파일
## `tricare_retriever_exp.ipynb`

## 목적
- `tricare_rag_integrated5.ipynb`의 MMR 단일 검색과
  **BM25 + Hybrid + Rerank** 파이프라인의 성능을 같은 질문으로 비교
- 성능 비교 기준: Faithfulness(충실도), Context Precision(정밀도), Hit Rate(히트율)
- 좋은 결과를 확인한 후 integrated5에 반영 예정

## 실험 섹션 구성
| 섹션 | 내용 |
|---|---|
| Section 0~11 | integrated5와 동일 (환경·DB·체인 구성) |
| **Section EX-1** | BM25Retriever 단독 설정 및 테스트 |
| **Section EX-2** | Hybrid Search (BM25 + MMR 결합) |
| **Section EX-3** | Reranker (CrossEncoder) 추가 |
| **Section EX-4** | 성능 비교 — 같은 질문으로 4가지 방식 결과 비교 |
| **Section EX-5** | 정량 평가 (Hit Rate / MRR / Faithfulness) |

## 성능 판단 기준
```
정량 지표:
  Hit Rate   : 정답 문서가 검색 결과 k개 안에 들어있는 비율
  MRR        : 정답 문서의 평균 순위 역수 (1위=1.0, 2위=0.5, ...)
  Faithfulness: 답변이 검색된 문서에 근거하는 비율 (RAGAS 또는 LLM-as-judge)

정성 지표:
  retrieved_docs 출처 다양성 (같은 파일만 나오는지 vs 여러 파일에서 나오는지)
  답변에 Group A/B, 플랜명, 출처 파일명이 정확히 포함되는지
```


---
## Section 0 · 패키지 설치

### 개요
- 처음 실행 시에만 필요함
- 주석 해제 후 실행하면 됨


In [1]:
# !pip install langchain langchain-community langchain-chroma langchain-openai \
# pymupdf pdfplumber sentence-transformers chromadb python-dotenv -q


---
## Section 1 · 환경 설정

### 개요
- `.env` 파일에서 `OPENAI_API_KEY`를 읽어옴
- 벡터 DB 경로·컬렉션명을 상수로 분리해 관리함
  - `PERSIST_TEXT` / `COLLECTION_TEXT`: 텍스트+CSV 통합 저장소
  - `PERSIST_TABLE` / `COLLECTION_TABLE`: 비용 표 전용 저장소
- 팀 통합 벡터 DB 병합 시 이 상수들을 기준으로 경로를 맞춤


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('OPENAI_API_KEY')
print(f"✅ API Key 로드: {'성공' if api_key else '실패 → .env 파일 확인 필요'}")

# 벡터 DB 경로·컬렉션명 상수
# 팀 통합 시 이 두 상수만 보면 어느 팀원 DB인지 바로 알 수 있도록 tricare_ 프리픽스 유지
PERSIST_TEXT = './chroma_db'
PERSIST_TABLE= './chroma_db2'   # 표 내용도 들어감
COLLECTION_TEXT  = 'tricare_rag'
COLLECTION_TABLE = 'tricare_cost_tables'


✅ API Key 로드: 성공


---
## Section 2 · 파일 경로 설정

### 개요
- [변경] 기존 `PDF_COSTS`, `PDF_HANDBOOK` 개별 변수 방식 → `RAG_PDF_DOCS` 리스트 방식으로 교체함
  - 신규 PDF가 추가될 때 리스트에 경로 한 줄만 추가하면 되는 구조임
  - 유지보수성이 크게 향상됨
- `RAG_PDF_DOCS`: `chroma_db`(텍스트 저장소)에 들어갈 PDF 목록
- `TABLE_PDF_DOCS`: `chroma_db2`(표 전용 저장소)에 추가로 표 추출할 PDF 목록
  - `Costs_Fees.pdf`는 두 리스트 모두에 포함됨 (텍스트 + 표 이중 수록 의도적)
  - `TRICARE_ADDP_HB.pdf`는 비용 표가 없으므로 `TABLE_PDF_DOCS`에서 제외함
- 경로는 본인 로컬 환경에 맞게 수정할 것

### 신규 추가 파일 (integrated2 대비)
| 파일 | 저장소 | 이유 |
|---|---|---|
| `Overseas_HB.pdf` | RAG만 | 해외 절차·한국 Near Patient Program 등 서술형 |
| `Pharmacy_HB.pdf` | RAG + TABLE | 본문 서술 + 비용표 모두 존재 |
| `NGR_HB.pdf` | RAG + TABLE | 본문 서술 + 활성화 상태별 플랜 옵션 표 |
| `TRICARE_ADDP_HB.pdf` | RAG만 | ADSM 비용 없는 프로그램, 비용 표 없음 |
| `ADDP_Brochure.pdf` | RAG만 | 개요 서술 |
| `Maternity_Br.pdf` | RAG만 | 서술형 브로슈어 |
| `Retiring_AD_Br.pdf` | RAG만 | 서술형 브로슈어 |
| `Retiring_NGR_Br.pdf` | RAG만 | 서술형 브로슈어 |
| `QLEs_FS.pdf` | RAG만 | 서술형 팩트시트 |

In [3]:
import os

# ── 경로 설정 ──
# DATA_DIR: 로컬 PC의 PDF/CSV 파일이 들어있는 폴더 경로
# 이 변수 하나만 바꾸면 아래 모든 경로가 자동으로 바뀜
DATA_DIR = r'C:\SKN_24\3차 프로젝트\Care-Insure-Bot\data'

# ── RAG_PDF_DOCS ──
# chroma_db(tricare_rag)에 텍스트로 들어갈 PDF 목록
# 한국/OCONUS 적용 가능한 문서만 포함
# BOTH = 미국·해외 공통 / OCONUS = 해외 전용
# (CONUS 전용 문서 제외: Choices_US_HB, Retiring_AD_Br)
RAG_PDF_DOCS = [
    #  비용 (BOTH) 
    os.path.join(DATA_DIR, 'Costs_Fees.pdf'),

    #  해외 전용 핸드북 (OCONUS) 
    os.path.join(DATA_DIR, 'Overseas_HB(해외 프로그램 안내서).pdf'),

    #  약국 (BOTH: 해외 약국 섹션 포함) 
    os.path.join(DATA_DIR, 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf'),

    #  예비군·방위군 (BOTH: 활성화 시 해외 적용) 
    os.path.join(DATA_DIR, 'NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf'),

    #  평생 트라이케어 (BOTH: 해외 TFL 규정 포함) 
    os.path.join(DATA_DIR, 'TFL_HB(평생 트라이케어).pdf'),

    #  치과 (BOTH: OCONUS ADDP 섹션 포함) 
    os.path.join(DATA_DIR, 'TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf'),
    os.path.join(DATA_DIR, 'ADDP_Brochure_FINAL_122624_508c.pdf'),

    #  임산부 (BOTH: 해외 출산 규정 포함) 
    os.path.join(DATA_DIR, 'Maternity_Br (1).pdf'),

    #  Medicare 전환 (BOTH: 해외 거주자 Medicare 규정 포함)
    #os.path.join(DATA_DIR, 'Medicare_Turning_65_Br.pdf'),
    #os.path.join(DATA_DIR, 'Medicare_Under_65_Br (7).pdf'),

    #  예비군 전역 (BOTH: TRR→TFL 전환 포함)
    os.path.join(DATA_DIR, 'Retiring_NGR_Br.pdf'),

    # 생애사건 (BOTH: 해외 이사·command sponsorship 포함)
    os.path.join(DATA_DIR, 'QLEs_FS (2).pdf'),
]

# TABLE_PDF_DOCS
# chroma_db2(tricare_cost_tables)에 표(pdfplumber)로 들어갈 PDF 목록
# 실제 비용 수치 표가 있는 파일만 포함
TABLE_PDF_DOCS = [
    os.path.join(DATA_DIR, 'Costs_Fees.pdf'),
    os.path.join(DATA_DIR, 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf'),
    os.path.join(DATA_DIR, 'NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf'),
    os.path.join(DATA_DIR, 'TFL_HB(평생 트라이케어).pdf'),
]

#  CSV 경로 
CSV_MENTAL= os.path.join(DATA_DIR, 'mental_health_services.csv')
CSV_COSTS = os.path.join(DATA_DIR, 'Health_Plan_Costs.csv')
CSV_PLANS = os.path.join(DATA_DIR, 'TricarePlans.csv')
CSV_EXCLUSIONS = os.path.join(DATA_DIR, 'tricare_exclusions.csv')

#  파일 존재 여부 확인 
print('[ RAG PDF 목록 ]')
for path in RAG_PDF_DOCS:
    status = '✅' if os.path.exists(path) else '❌ 없음 →'
    print(f'  {status} {os.path.basename(path)}')

print('\n[ TABLE PDF 목록 ]')
for path in TABLE_PDF_DOCS:
    status = '✅' if os.path.exists(path) else '❌ 없음 →'
    print(f'  {status} {os.path.basename(path)}')

print('\n[ CSV 목록 ]')
for path in [CSV_MENTAL, CSV_COSTS, CSV_PLANS, CSV_EXCLUSIONS]:
    status = '✅' if os.path.exists(path) else '❌ 없음 →'
    print(f'  {status} {os.path.basename(path)}')


[ RAG PDF 목록 ]
  ✅ Costs_Fees.pdf
  ✅ Overseas_HB(해외 프로그램 안내서).pdf
  ✅ Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
  ✅ NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf
  ✅ TFL_HB(평생 트라이케어).pdf
  ✅ TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf
  ✅ ADDP_Brochure_FINAL_122624_508c.pdf
  ✅ Maternity_Br (1).pdf
  ✅ Retiring_NGR_Br.pdf
  ✅ QLEs_FS (2).pdf

[ TABLE PDF 목록 ]
  ✅ Costs_Fees.pdf
  ✅ Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
  ✅ NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf
  ✅ TFL_HB(평생 트라이케어).pdf

[ CSV 목록 ]
  ✅ mental_health_services.csv
  ✅ Health_Plan_Costs.csv
  ✅ TricarePlans.csv
  ✅ tricare_exclusions.csv


---
## Section 3 · 임베딩 모델 초기화 — `BAAI/bge-m3`

### 개요
- 100개 이상의 언어를 동일한 벡터 공간에 매핑하는 다국어 임베딩 모델임
- 한국어 질문과 영어 문서 간 유사도 매칭이 가능함
- 팀원 Cigna 코드와 동일한 모델을 사용해 임베딩 방식을 통일함
- GPU 있으면 `device: cuda`로 변경하면 속도가 빨라짐


In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'cpu'}  # GPU 있으면 'cuda'로 변경
)

print('✅ BAAI/bge-m3 임베딩 모델 로드 완료')


C:\Users\kjw70\AppData\Local\Temp\ipykernel_4256\2816061350.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(



✅ BAAI/bge-m3 임베딩 모델 로드 완료


---
## Section 4 · 전처리 유틸 함수 + PDF 로드

### 변경 사항
- `clean_text()`: PDF 특수문자·중복 공백 정제
- `normalize_cell()`: 표 체크마크 → Covered/Not covered 텍스트 변환
- `is_noise_line()`: 반복 헤더·URL 등 노이즈 행 제거
- `is_oconus_relevant()`: **[신규]** BOTH 문서에서 한국/해외 관련 페이지만 필터링
- `enrich_tricare_text()`: 본문 뒤에 search_tags 블록 추가
- `LOCATION_MAP`: 파일명 → OCONUS/BOTH 태그 매핑

### 페이지 필터링 로직
```
OCONUS 문서 → 전 페이지 사용 (이미 해외 전용)
BOTH 문서   → OCONUS_KEYWORDS 포함 페이지만 사용
CONUS 문서  → RAG_PDF_DOCS에서 이미 제외
```


In [5]:
#from langchain_community.document_loaders import PyMuPDFLoader
#
#all_pdf_docs = []
#
## RAG_PDF_DOCS 리스트를 순회하며 일괄 로드
## 신규 PDF가 추가되어도 Section 2의 리스트만 수정하면 이 셀은 수정 불필요
#for pdf_path in RAG_PDF_DOCS:
#    docs = PyMuPDFLoader(pdf_path).load()
#    # 파일명을 source_file 메타데이터로 추가해 팀 통합 시 출처 구분 용이하게 함
#    for doc in docs:
#        doc.metadata['source_file'] = os.path.basename(pdf_path)
#    all_pdf_docs.extend(docs)
#    print(f'  ✅ {os.path.basename(pdf_path)}: {len(docs)}페이지')
#
#print(f'\n📄 총 PDF 페이지 합계: {len(all_pdf_docs)}페이지')
#print(f'\n--- 샘플 메타데이터 ---')
#print(all_pdf_docs[0].metadata)


# 내용 수정: 일부 pdf - 보험 혜택 가이드라인 중 주한미군&한국 내 적용이 일부만 되는 것이 있기 때문

In [6]:
import re
from langchain_community.document_loaders import PyMuPDFLoader

# 전처리 유틸 함수
def clean_text(text: str) -> str:
    """
    PDF에서 추출한 텍스트의 불필요한 문자 제거
    - \xa0 (non-breaking space): PDF에서 자주 나오는 특수 공백
    - \u200b (zero-width space): 눈에 안 보이지만 검색에 영향 주는 문자
    - 연속 공백·탭 → 공백 1개로 통일
    - 3줄 이상 빈 줄 → 2줄로 통일
    """
    text = text.replace('\xa0', ' ')
    text = text.replace('\u200b', ' ')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def normalize_cell(cell: str) -> str:
    """
    표 셀의 체크마크·빈칸을 LLM이 읽기 좋은 텍스트로 변환
    - ✓/√/v/V/● → 'Covered'  (보장됨)
    - ✗/×/x/X/✘ → 'Not covered'  (보장 안 됨)
    - 빈칸        → 'N/A'
    """
    if not cell:
        return 'N/A'
    cell = cell.strip()
    if cell in ['✓', '√', 'v', 'V', '●', 'Yes', 'yes', 'Y']:
        return 'Covered'
    if cell in ['✗', '×', 'x', 'X', '✘', 'No', 'no', 'N']:
        return 'Not covered'
    return cell


def is_noise_line(line: str) -> bool:
    """
    반복 헤더·URL·업데이트 날짜 등 RAG에 불필요한 노이즈 행 판별
    True를 반환하면 해당 줄은 청크에서 제거됨
    """
    noise_patterns = [
        r'^group a.*group b',
        r'^covered service.*group',
        r'^are you in group',
        r'^this is an overview',
        r'^visit www\.tricare',
        r'^for more information.*go to',
        r'^updated (january|february|march|april|may|june|july|august|september|october|november|december)',
    ]
    return any(re.search(p, line.lower()) for p in noise_patterns)


# OCONUS 관련성 키워드 
# BOTH 문서에서 이 키워드 중 하나라도 포함된 페이지만 남김
# 한국/해외 거주자에게 해당하는 내용만 파싱하기 위한 필터
OCONUS_KEYWORDS = [
    'overseas', 'oconus', 'outside the continental',
    'south korea', 'korea', 'usfk',
    'outside the u.s.', 'outside the united states',
    'international', 'host nation', 'foreign country',
    'tricare prime overseas', 'tricare select overseas',
    'tricare prime remote overseas',
    'near patient', 'overseas claim',
]

def is_oconus_relevant(text: str) -> bool:
    """
    페이지 텍스트에 해외/한국 관련 키워드가 있는지 확인
    - OCONUS 문서: 필터 없이 전 페이지 사용
    - BOTH 문서:   이 함수로 걸러서 관련 페이지만 사용
    - CONUS 문서:  RAG_PDF_DOCS에서 이미 제외됨
    """
    text_lower = text.lower()
    return any(kw in text_lower for kw in OCONUS_KEYWORDS)


# TRICARE_SEARCH_TAGS 
# 각 청크 본문 뒤에 붙이는 검색 향상용 키워드 블록
# bge-m3 임베딩에서 한국어 질문 → 영어 문서 유사도 매칭률을 높이는 역할
TRICARE_SEARCH_TAGS = [
    'coverage', 'covered', 'not covered', 'benefit', 'cost', 'copay',
    'deductible', 'enrollment fee', 'cost-share', 'prior authorization',
    'TRICARE Prime', 'TRICARE Select', 'TRICARE For Life',
    'Group A', 'Group B', 'active duty', 'retiree', 'reserve',
    'OCONUS', 'overseas', 'USFK', '주한미군',
    '보장', '비용', '본인부담금', '사전승인', '해외',
]


def enrich_tricare_text(text: str, fname: str) -> str:
    """
    본문 뒤에 [search_tags] 블록 추가
    - insurer: TRICARE → 팀 통합 시 어느 보험사 문서인지 구분
    - location: OCONUS/BOTH → 검색 필터링에 사용
    - keywords: 도메인 키워드 → 한/영 유사도 향상
    이 블록은 벡터 임베딩에는 포함되지만 LLM 컨텍스트에는 제거 후 전달됨
    (format_docs 함수에서 제거)
    """
    location = LOCATION_MAP.get(fname, 'BOTH')
    tags = '\n'.join([
        '[search_tags]',
        'insurer: TRICARE',
        f'location: {location}',
        'keywords: ' + ', '.join(TRICARE_SEARCH_TAGS),
    ])
    return f'{text}\n\n{tags}'



# LOCATION_MAP
# 파일명 → location 태그 매핑
# - OCONUS: 해외 전용 문서 → 전 페이지 사용
# - CONUS:  미국 내 전용 → RAG_PDF_DOCS에서 이미 제외됨
# - BOTH:   공통 문서 → OCONUS 키워드 있는 페이지만 사용
# 매핑에 없는 파일은 자동으로 'BOTH' 처리됨
LOCATION_MAP = {
    'Overseas_HB(해외 프로그램 안내서).pdf': 'OCONUS',
}
# 나머지 파일은 모두 BOTH (기본값)
# CONUS 전용 파일은 RAG_PDF_DOCS에서 이미 제외함



# PDF 로드 메인 루프
all_pdf_docs = []
skipped_pages = 0

for pdf_path in RAG_PDF_DOCS:
    docs  = PyMuPDFLoader(pdf_path).load()
    fname = os.path.basename(pdf_path)
    loc   = LOCATION_MAP.get(fname, 'BOTH')

    kept = []
    for doc in docs:
        cleaned = clean_text(doc.page_content)

        # OCONUS 문서: 전 페이지 사용
        # BOTH 문서:   OCONUS 키워드 있는 페이지만 사용
        if loc == 'BOTH' and not is_oconus_relevant(cleaned):
            skipped_pages += 1
            continue

        # 노이즈 행 제거
        lines   = [ln for ln in cleaned.split('\n') if not is_noise_line(ln)]
        cleaned = '\n'.join(lines)

        # search_tags 추가
        doc.page_content        = enrich_tricare_text(cleaned, fname)
        doc.metadata['source_file'] = fname
        doc.metadata['location']    = loc
        kept.append(doc)

    all_pdf_docs.extend(kept)
    print(f'  ✅ {fname} [{loc}]: {len(docs)}페이지 중 {len(kept)}페이지 사용')

print(f'\n📄 총 사용 페이지: {len(all_pdf_docs)}  (필터로 제외된 페이지: {skipped_pages})')
print(f'\n--- 샘플 메타데이터 ---')
print(all_pdf_docs[0].metadata)
print(f'\n--- 샘플 page_content (앞 400자) ---')
print(all_pdf_docs[0].page_content[:400])

# Medicare_Turning_65_Br.pdf 파일 제외해서 오류 뜨는데 일부로 안 넣은 거라 무시


  ✅ Costs_Fees.pdf [BOTH]: 4페이지 중 3페이지 사용
  ✅ Overseas_HB(해외 프로그램 안내서).pdf [OCONUS]: 16페이지 중 16페이지 사용
  ✅ Pharmacy_HB(tricare 약국 프로그램 안내서).pdf [BOTH]: 32페이지 중 14페이지 사용
  ✅ NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf [BOTH]: 16페이지 중 4페이지 사용
  ✅ TFL_HB(평생 트라이케어).pdf [BOTH]: 44페이지 중 20페이지 사용
  ✅ TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf [BOTH]: 24페이지 중 16페이지 사용
  ✅ ADDP_Brochure_FINAL_122624_508c.pdf [BOTH]: 4페이지 중 3페이지 사용
  ✅ Maternity_Br (1).pdf [BOTH]: 4페이지 중 4페이지 사용
  ✅ Retiring_NGR_Br.pdf [BOTH]: 4페이지 중 3페이지 사용
  ✅ QLEs_FS (2).pdf [BOTH]: 4페이지 중 3페이지 사용

📄 총 사용 페이지: 86  (필터로 제외된 페이지: 66)

--- 샘플 메타데이터 ---
{'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2026-01-06T16:05:56-05:00', 'source': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\Costs_Fees.pdf', 'file_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\Costs_Fees.pdf', 'total_pages': 4, 'format': 'PDF 1.6', 'title': 'TRICARE 2026 Costs and Fees Sheet', 'author': 'TRICARE Co

---
## Section 5 · PDF 청킹 — 단락 단위 분리

### 개요
- [변경] RecursiveCharacterTextSplitter → **단락(`\n\n`) 단위 직접 분리**로 교체
  - 청크 경계가 문장 중간에서 잘리는 문제 방지
  - 80자 이상 단락만 Document로 생성 (너무 짧은 단락 제거)
- RecursiveCharacterTextSplitter는 CSV 청킹에만 재사용됨 (Section 6)
- 단락 분리 후에도 `source_file`, `location` 메타데이터는 그대로 유지됨


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document as LCDocument

# CSV 청킹용 text_splitter는 그대로 유지 (Section 6에서 재사용)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

# PDF는 단락(\n\n) 단위 직접 분리 — 문장 중간 절단 방지
# 80자 미만 단락은 헤더·페이지 번호 등 노이즈일 가능성이 높으므로 제거
MIN_CHUNK_CHARS = 80

pdf_chunks = []
for doc in all_pdf_docs:
    # search_tags 블록은 분리하지 않고 단락 분리 전에 제거 후 각 청크에 다시 붙임
    # (search_tags가 모든 청크에 포함되어야 유사도 향상 효과가 있음)
    full_text = doc.page_content
    if '[search_tags]' in full_text:
        main_text, tags_block = full_text.split('[search_tags]', 1)
        tags_block = '[search_tags]' + tags_block
    else:
        main_text  = full_text
        tags_block = ''

    paragraphs = [p.strip() for p in main_text.split('\n\n') if len(p.strip()) >= MIN_CHUNK_CHARS]
    for para in paragraphs:
        chunk_text = para + ('\n\n' + tags_block if tags_block else '')
        pdf_chunks.append(LCDocument(
            page_content=chunk_text,
            metadata=doc.metadata.copy()
        ))

print(f'분할 결과: {len(all_pdf_docs)}페이지 → {len(pdf_chunks)}개 청크')
print(f'(단락 단위 분리, {MIN_CHUNK_CHARS}자 미만 제거)')
print(f'\n--- 청크 샘플 ---')
print(pdf_chunks[0].page_content[:400])
print(f'\nMetadata: {pdf_chunks[0].metadata}')


분할 결과: 86페이지 → 86개 청크
(단락 단위 분리, 80자 미만 제거)

--- 청크 샘플 ---
TRICARE® 2026 Costs and Fees
TRICARE For Life, survivors, and medically retired individuals, visit www.tricare.mil/comparecosts. 
• You’re in Group A if your initial enlistment or appointment or that of your uniformed services sponsor began before Jan. 1, 2018. 
• You’re in Group B if your initial enlistment or appointment or that of your uniformed services sponsor began on or after Jan. 1, 2018.


Metadata: {'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 19.5 (Macintosh)', 'creationdate': '2026-01-06T16:05:56-05:00', 'source': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\Costs_Fees.pdf', 'file_path': 'C:\\SKN_24\\3차 프로젝트\\Care-Insure-Bot\\data\\Costs_Fees.pdf', 'total_pages': 4, 'format': 'PDF 1.6', 'title': 'TRICARE 2026 Costs and Fees Sheet', 'author': 'TRICARE Communications', 'subject': '', 'keywords': '', 'moddate': '2026-01-28T17:23:05-05:00', 'trapped': '', 'modDate': "D:20260128172305-05'00'", 'creati

---
## Section 6 · CSV 로드 — 텍스트 벡터 저장소용 3종

### 개요
- integrated2와 동일한 방식으로 유지함
- `CSVLoader` 대신 직접 변환 방식을 사용함
  - 청킹 후에도 서비스명·항목명이 각 청크에 남아 출처 추적이 가능함

### 3종 파일별 변환 방식
| 파일 | page_content 구성 | metadata 키 |
|---|---|---|
| `mental_health_services.csv` | `서비스: {서비스}\n내용: {내용}` | `source`, `서비스명`, `type` |
| `TricarePlans.csv` | `항목: {Plans}\n플랜명: {플랜명}\n내용: {내용}` | `source`, `plan_name`, `type` |
| `tricare_exclusions.csv` | `제외항목: {name}\n내용: {content}` | `source`, `exclusion_name`, `type`, `url` |


In [8]:
import pandas as pd
from langchain_core.documents import Document

#  mental_health_services.csv 
# 정신건강 서비스별 보장 내용
# 서비스명을 page_content에 직접 포함 → 청킹 후에도 서비스명이 남아 출처 추적 가능
df_mental  = pd.read_csv(CSV_MENTAL, encoding='utf-8-sig')
mental_docs = []
for _, row in df_mental.iterrows():
    mental_docs.append(Document(
        page_content=f"서비스: {row['서비스']}\n내용: {row['내용']}",
        metadata={
            'source':   'mental_health_services.csv',
            '서비스명': row['서비스'],
            'type':     'mental_health',
            'location': 'BOTH',  # CSV는 플랜 공통 내용이므로 BOTH
        }
    ))

mental_chunks = text_splitter.split_documents(mental_docs)
print(f'📋 mental_health_services: {len(df_mental)}행 → {len(mental_chunks)}청크')


📋 mental_health_services: 8행 → 12청크


In [9]:
# TricarePlans.csv 
# 플랜별 특징 비교 (컬럼 구조: Plans 항목명 + 각 플랜명)
# 플랜별로 항목-내용 쌍을 Document로 변환 → 플랜 비교 질문에 검색되도록 함
df_plans  = pd.read_csv(CSV_PLANS, encoding='utf-8-sig')
plan_cols = [c for c in df_plans.columns if c != 'Plans']
plans_docs = []
for _, row in df_plans.iterrows():
    topic = row['Plans']
    for plan_name in plan_cols:
        content = str(row[plan_name]).strip()
        if not content or content == 'nan':
            continue
        plans_docs.append(Document(
            page_content=f"항목: {topic}\n플랜명: {plan_name}\n내용: {content}",
            metadata={
                'source':    'TricarePlans.csv',
                'plan_name': plan_name,
                'type':      'plan_info',
                'location':  'BOTH',  # 플랜 정보는 공통
            }
        ))

plans_chunks = text_splitter.split_documents(plans_docs)
print(f'📋 TricarePlans: {len(df_plans)}행 × {len(plan_cols)}플랜 → {len(plans_chunks)}청크')


📋 TricarePlans: 10행 × 16플랜 → 164청크


In [10]:
#  tricare_exclusions.csv 
# TRICARE 보장 제외 항목 (tricare.mil 크롤링 → CSV 저장본)
# url은 metadata에만 저장하여 LLM 컨텍스트에는 노출하지 않음
df_excl = pd.read_csv(CSV_EXCLUSIONS, encoding='utf-8-sig')
exclusion_docs = []
for _, row in df_excl.iterrows():
    exclusion_docs.append(Document(
        page_content=f"제외항목: {row['name']}\n내용: {row['content']}",
        metadata={
            'source':         'tricare_exclusions.csv',
            'exclusion_name': row['name'],
            'type':           'exclusion',
            'url':            row['url'],
            'location':       'BOTH',  # 제외 항목은 플랜 공통
        }
    ))

exclusion_chunks = text_splitter.split_documents(exclusion_docs)
print(f'📋 tricare_exclusions: {len(df_excl)}행 → {len(exclusion_chunks)}청크')


📋 tricare_exclusions: 55행 → 64청크


---
## Section 7 · 텍스트+CSV 통합 벡터 저장소 생성 (`chroma_db`)

### 개요
- PDF 텍스트 + CSV 3종(정신건강 / 플랜 정보 / 제외 항목)을 하나의 저장소에 통합함
- Retriever가 소스 구분 없이 가장 관련 있는 청크를 한 번에 검색함
- `metadata.source`와 `metadata.type`으로 청크 출처는 항상 추적 가능함

### 통합 청크 구성
| 소스 | 청크 변수 |
|---|---|
| PDF 텍스트 15종 | `pdf_chunks` |
| mental_health_services.csv | `mental_chunks` |
| TricarePlans.csv | `plans_chunks` |
| tricare_exclusions.csv | `exclusion_chunks` |


In [11]:
from langchain_chroma import Chroma

# PDF + CSV 3종 모두 합산
all_text_chunks = pdf_chunks + mental_chunks + plans_chunks + exclusion_chunks

print(f'📦 청크 합산')
print(f'PDF 텍스트 ({len(RAG_PDF_DOCS)}종):      {len(pdf_chunks)}개')
print(f'mental_health_services: {len(mental_chunks)}개')
print(f'TricarePlans:           {len(plans_chunks)}개')
print(f'tricare_exclusions:     {len(exclusion_chunks)}개')
print(f'합계:                   {len(all_text_chunks)}개')
print('⏳ 벡터 저장소 생성 중...')

# PERSIST_TEXT / COLLECTION_TEXT는 Section 1에서 정의한 상수
vector_store = Chroma.from_documents(
    documents=all_text_chunks,
    embedding=embedding_model,
    collection_name=COLLECTION_TEXT,
    persist_directory=PERSIST_TEXT
)

print(f'✅ 텍스트 벡터 저장소 생성 완료: {vector_store._collection.count()}개 벡터')


📦 청크 합산
PDF 텍스트 (10종):      86개
mental_health_services: 12개
TricarePlans:           164개
tricare_exclusions:     64개
합계:                   326개
⏳ 벡터 저장소 생성 중...
✅ 텍스트 벡터 저장소 생성 완료: 326개 벡터


---
## Section 8 · PDF 표 추출 — pdfplumber + `Health_Plan_Costs.csv`

### 개요
- [변경] 기존 `Costs_Fees.pdf` 단일 대상 → `TABLE_PDF_DOCS` 리스트 순회 방식으로 확장함
  - `Pharmacy_HB.pdf`, `NGR_HB.pdf`, `TFL_HB.pdf`, `Plans_Overview_FS.pdf` 표 추가 수록
- `pdfplumber`는 표의 셀 좌표를 인식해 `행 | 열` 구조를 보존함
- 표는 텍스트 벡터 저장소와 분리해 별도 저장함
  - 비용 질문 시 텍스트 청크가 섞이면 노이즈가 될 수 있음
- `Health_Plan_Costs.csv`는 행 자체가 비용 단위로 구성되어 있어 추가 청킹 불필요함


In [12]:
import pdfplumber

def extract_tables_from_pdf(pdf_path: str) -> list:
    """
    pdfplumber로 PDF 전체에서 표를 추출함.
    [변경] normalize_cell() 적용 — ✓/✗ → Covered/Not covered 정규화
    [변경] 빈 행 제거 — 모든 셀이 N/A인 행은 노이즈이므로 제외
    각 셀은 ' | '로 구분되어 LLM이 행/열 관계를 파악할 수 있음.
    source_file 메타데이터로 어느 PDF에서 왔는지 팀 통합 시 구분 가능함.
    """
    table_docs = []
    file_name  = os.path.basename(pdf_path)
    loc        = LOCATION_MAP.get(file_name, 'BOTH')
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for table_idx, table in enumerate(tables):
                if not table or len(table) < 2:
                    continue
                lines = []
                for row in table:
                    # normalize_cell 적용 — ✓/✗/빈칸을 텍스트로 변환
                    cleaned_row = [
                        normalize_cell(str(cell).strip().replace('\n', ' ') if cell else '')
                        for cell in row
                    ]
                    # 모든 셀이 N/A인 행은 빈 행이므로 제거
                    if all(c == 'N/A' for c in cleaned_row):
                        continue
                    lines.append(' | '.join(cleaned_row))
                if not lines:
                    continue
                table_docs.append(LCDocument(
                    page_content=f'[TRICARE 비용표 | {file_name} | p.{page_num + 1}]\n' + '\n'.join(lines),
                    metadata={
                        'source':      pdf_path,
                        'source_file': file_name,
                        'page':        page_num + 1,
                        'type':        'cost_table',
                        'table_idx':   table_idx + 1,
                        'location':    loc,
                    }
                ))
    return table_docs

# TABLE_PDF_DOCS 리스트를 순회하며 일괄 표 추출
table_docs_pdf = []
for pdf_path in TABLE_PDF_DOCS:
    docs = extract_tables_from_pdf(pdf_path)
    table_docs_pdf.extend(docs)
    print(f'  📊 {os.path.basename(pdf_path)}: {len(docs)}개 표 추출')

print(f'\n📊 PDF 표 합계: {len(table_docs_pdf)}개')
print('\n--- PDF 표 샘플 ---')
if table_docs_pdf:
    print(table_docs_pdf[0].page_content[:300])


  📊 Costs_Fees.pdf: 8개 표 추출
  📊 Pharmacy_HB(tricare 약국 프로그램 안내서).pdf: 8개 표 추출
  📊 NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf: 14개 표 추출
  📊 TFL_HB(평생 트라이케어).pdf: 14개 표 추출

📊 PDF 표 합계: 44개

--- PDF 표 샘플 ---
[TRICARE 비용표 | Costs_Fees.pdf | p.1]
ADSMs, ADFMs, and transitional survivors | N/A | N/A
Covered service | Group A | Group B
All covered services | $0 | $0
Retirees, their family members, and all others | N/A | N/A
Covered service | Group A | Group B
Preventive care visit | $0 | $0
Primary care out


In [13]:
# Health_Plan_Costs.csv — 비용 전용 저장소용
# 탭(플랜·그룹 정보) + 비용 유형 + 플랜 1/2 수치를 하나의 page_content로 구성
# pdfplumber 표와 동일한 저장소에 넣어 비용 질문의 검색 커버리지를 높임
df_costs = pd.read_csv(CSV_COSTS, encoding='utf-8-sig')
cost_docs = []
for _, row in df_costs.iterrows():
    tab       = str(row['탭']).strip()
    cost_type = str(row['비용 유형']).strip()
    plan1     = str(row['플랜 1']).strip()
    plan2     = str(row['플랜 2']).strip() if str(row['플랜 2']) != 'nan' else ''

    content = f"플랜/그룹: {tab}\n비용 유형: {cost_type}\n플랜 1: {plan1}"
    if plan2:
        content += f"\n플랜 2: {plan2}"

    cost_docs.append(Document(
        page_content=content,
        metadata={
            'source': 'Health_Plan_Costs.csv',
            'cost_type': cost_type,
            'type': 'cost_table'
        }
    ))

# Health_Plan_Costs는 행 자체가 비용 단위로 구성되어 있어 추가 청킹 불필요함
print(f'📊 Health_Plan_Costs: {len(df_costs)}행 → {len(cost_docs)}개 Document')
print('\n--- CSV 비용 샘플 ---')
print(cost_docs[0].page_content)
print(cost_docs[1].page_content)
print(cost_docs[2].page_content)
print(cost_docs[3].page_content)
print(cost_docs[4].page_content)


📊 Health_Plan_Costs: 234행 → 234개 Document

--- CSV 비용 샘플 ---
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Enrollment fee (annual)
플랜 1: $0
플랜 2: $0
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Deductible (annual)
플랜 1: $0
플랜 2: E-1 through E-4: $50 per individual $100 per family E-5 & above: $150 per individual $300 per family
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Catastrophic cap (annual)
플랜 1: $1,000 per family
플랜 2: $1,000 per family
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Clinical preventive services
플랜 1: Network: $0 Non-network: Point-of-service fees
플랜 2: Network and Non-network: $0
플랜/그룹: TRICARE Prime & TRICARE Select Active duty family members & transitional survivors Group A
비용 유형: Outpatient visit — primary care
플랜 1: Network:

---
## Section 9 · 표 전용 벡터 저장소 생성 (`chroma_db2`)

### 개요
- PDF 표 + `Health_Plan_Costs.csv`를 하나의 표 전용 저장소에 통합함
- `k=3`: 표 데이터는 정보 밀도가 높아 3개면 충분함
- 비용 질문 시 텍스트 청크가 섞이지 않아 수치 답변 정확도가 높아짐

### 재실행 시 벡터 저장소 불러오는 방법
```python
vector_store = Chroma(
    collection_name='tricare_rag',
    embedding_function=embedding_model,
    persist_directory='./chroma_db'
)
table_vector_store = Chroma(
    collection_name='tricare_cost_tables',
    embedding_function=embedding_model,
    persist_directory='./chroma_db2'
)
```


In [14]:
# PDF 표 + CSV 비용 Document 합산
all_table_docs = table_docs_pdf + cost_docs

print(f'📦 표 데이터 합산')
print(f'   PDF 표 (pdfplumber, {len(TABLE_PDF_DOCS)}종): {len(table_docs_pdf)}개')
print(f'   Health_Plan_Costs.csv:              {len(cost_docs)}개')
print(f'   합계:                               {len(all_table_docs)}개')
print('⏳ 표 전용 벡터 저장소 생성 중...')

# PERSIST_TABLE / COLLECTION_TABLE은 Section 1에서 정의한 상수
table_vector_store = Chroma.from_documents(
    documents=all_table_docs,
    embedding=embedding_model,
    collection_name=COLLECTION_TABLE,
    persist_directory=PERSIST_TABLE
)

print(f'✅ 표 전용 벡터 저장소 생성 완료: {table_vector_store._collection.count()}개 벡터')


📦 표 데이터 합산
   PDF 표 (pdfplumber, 4종): 44개
   Health_Plan_Costs.csv:              234개
   합계:                               278개
⏳ 표 전용 벡터 저장소 생성 중...
✅ 표 전용 벡터 저장소 생성 완료: 278개 벡터


---
## Section 10 · Retriever 설정 — MMR 검색

### 개요
- [변경] `similarity` → **MMR(Maximal Marginal Relevance)** 검색으로 교체
  - 중복 청크 감소, 다양한 문서에서 검색 가능
  - `fetch_k=20`으로 20개 후보 중 다양성 기준으로 k=6 선택
- 텍스트 retriever: k=6, fetch_k=20
- 표 retriever: k=3, fetch_k=10 (표 데이터는 밀도 높아 3개면 충분)
- `normalize_question()`: LLM이 질문에서 intent/region/영어쿼리를 추출해 검색 정확도 향상
  - 비용이 발생하므로 RAG 검색 단계에서만 호출


In [15]:
from langchain_openai import ChatOpenAI

# 팀원 출력 방식으로 통일
# gpt-4o-mini 호출 비용 발생 → RAG 체인 내부에서만 호출
_norm_model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

NORMALIZE_PROMPT = """
You are a TRICARE insurance query analyzer.
Given the user's question, extract:
1. intent: one of [coverage, eligibility, cost, pharmacy, dental, overseas, general]
2. region: one of [CONUS, OCONUS, korea, unknown]
3. english_query: rewrite the question in clear English (for vector search)

Respond ONLY in JSON format:
{{"intent": "...", "region": "...", "english_query": "..."}}

Question: {question}
"""

import json as _json

def normalize_question(question: str) -> dict:
    """질문에서 intent/region/영어쿼리 추출 — 검색 정확도 향상용"""
    try:
        resp = _norm_model.invoke(NORMALIZE_PROMPT.format(question=question))
        return _json.loads(resp.content)
    except Exception:
        # 파싱 실패 시 원문 질문으로 fallback
        return {'intent': 'general', 'region': 'unknown', 'english_query': question}


# MMR 검색 함수 — as_retriever 대신 max_marginal_relevance_search 직접 호출
# normalize_question으로 영어 쿼리 추출 → 검색 정확도 향상
def mmr_retrieve(question: str, store, k: int = 6, fetch_k: int = 20) -> list:
    """MMR 검색: fetch_k개 후보 중 다양성 기준으로 k개 선택"""
    norm   = normalize_question(question)
    query  = norm.get('english_query', question)
    region = norm.get('region', 'unknown')
    docs   = store.max_marginal_relevance_search(query, k=k, fetch_k=fetch_k)
    # OCONUS 질문이면 location 필터 적용 (CONUS 전용 문서 제외)
    if region in ('OCONUS', 'korea'):
        docs = [d for d in docs if d.metadata.get('location', 'BOTH') != 'CONUS'] or docs
    return docs


# 기존 as_retriever도 유지 (make_rag_chain 호환성)
retriever = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 6, 'fetch_k': 20}
)
table_retriever = table_vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 3, 'fetch_k': 10}
)

# 동작 확인
test_result = retriever.invoke('TRICARE Prime 본인부담금')
print(f'🔍 MMR retriever 테스트: {len(test_result)}개 검색됨')
for i, d in enumerate(test_result):
    print(f'  [{i+1}] {d.metadata.get("source_file", d.metadata.get("source", "?"))} | loc={d.metadata.get("location","?")} : {d.page_content[:60]}...')

# normalize_question 테스트
test_q = '한국에서 TRICARE Prime 외래 진료 본인부담금이 얼마예요?'
norm_result = normalize_question(test_q)
print(f'\n🔍 normalize_question 테스트:')
print(f'  intent:        {norm_result["intent"]}')
print(f'  region:        {norm_result["region"]}')
print(f'  english_query: {norm_result["english_query"]}')


🔍 MMR retriever 테스트: 6개 검색됨
  [1] Costs_Fees.pdf | loc=BOTH : TRICARE® 2026 Costs and Fees
TRICARE For Life, survivors, an...
  [2] Overseas_HB(해외 프로그램 안내서).pdf | loc=OCONUS : HEALTH CARE OPTIONS 
TRICARE Prime Overseas Options (continu...
  [3] TricarePlans.csv | loc=BOTH : 항목: Will I have to file my own claims?
플랜명: TRICARE Prime
내용...
  [4] TricarePlans.csv | loc=BOTH : 항목: Do I have to enroll? What is the annual fee?
플랜명: TRICAR...
  [5] TricarePlans.csv | loc=BOTH : 항목: How much do I pay for an outpatient visit?
플랜명: TRICARE ...
  [6] TricarePlans.csv | loc=BOTH : 항목: What is the maximum I'll pay out-of-pocket?
플랜명: TRICARE...

🔍 normalize_question 테스트:
  intent:        cost
  region:        korea
  english_query: What is the out-of-pocket cost for TRICARE Prime outpatient care in Korea?


---
## Section 11 · LLM + RAG 체인 구성

### 개요
- integrated2와 동일하게 유지함 (팀원 통일 사항)
- `gpt-4o-mini`, `temperature=0.1`: 보험 정보 오답은 실제 피해로 이어질 수 있어
  낮은 temperature로 문서 기반 답변을 유도함
- `detect_language()`: 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리함
  - 팀원 A(rag_utils.py)와 동일한 방식으로 통일함
- `make_rag_chain()`: 질문마다 언어를 감지해 프롬프트에 `answer_language`를 f-string으로 주입함
  - `retriever`를 인자로 받아 텍스트용 / 표 전용 체인 모두 이 함수 하나로 생성 가능함
- `format_docs()`: 출처 정보(`[source, p.페이지]`)를 청크 앞에 붙여 반환함
  - 팀원 RAG_TEMPLATE의 출처 인용 형식과 통일함

  **⬇️⬇️⬇️⬇️ 수정함***

  ## Section 11 · LLM + RAG 체인 구성

### 개요
- `gpt-4o-mini`, `temperature=0.1`: 보험 정보 오답은 실제 피해로 이어질 수 있어
  낮은 temperature로 문서 기반 답변을 유도함
- `detect_language()`: 유니코드 범위로 한/중/일을 판별하고 나머지는 영어로 처리함
  - 팀원 A(rag_utils.py)와 동일한 방식으로 통일함
- `make_rag_chain()`: 질문마다 언어를 감지해 프롬프트에 `answer_language`를 f-string으로 주입함
  - `retriever`를 인자로 받아 텍스트용 / 표 전용 체인 모두 이 함수 하나로 생성 가능함
- `format_docs()`: 출처 정보(`[source, p.페이지]`)를 청크 앞에 붙여 반환함
  - 팀원 RAG_TEMPLATE의 출처 인용 형식과 통일함
- [변경] OCONUS 컨텍스트 추가: 주한미군·한국 거주 수혜자 대상 시스템임을 프롬프트에 명시함
  - 한국에서는 트라이케어가 1차 지불자, Medicare 해외 적용 불가 등 핵심 규칙 주입


In [16]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(model='gpt-4o-mini', temperature=0.1)

# 언어 감지 함수 (팀원 통일 방식 그대로 유지)
def detect_language(text: str) -> str:
    if any('\u4e00' <= c <= '\u9fff' for c in text):
        return 'zh'
    if any('\u3040' <= c <= '\u30ff' for c in text):
        return 'ja'
    if any('\uac00' <= c <= '\ud7a3' for c in text):
        return 'ko'
    return 'en'

LANGUAGE_NAME_MAP = {
    'ko': 'Korean',
    'en': 'English',
    'zh': 'Chinese',
    'ja': 'Japanese',
    'es': 'Spanish',
    'fr': 'French',
    'de': 'German',
    'other': "the same language as the user's question",
}

# format_docs: 출처 정보(source_file, p.페이지)를 청크 앞에 붙임
# search_tags 블록은 LLM 컨텍스트에서 제거 — LLM에게 태그 텍스트가 보이면 노이즈
def format_docs(docs):
    result = []
    for doc in docs:
        source = doc.metadata.get('source_file', doc.metadata.get('source', 'unknown'))
        page   = doc.metadata.get('page', '')
        label  = f'[{source}{", p." + str(page) if page else ""}]'
        # search_tags 블록 제거 후 LLM에 전달
        content = doc.page_content
        if '[search_tags]' in content:
            content = content.split('[search_tags]')[0].strip()
        result.append(f'{label}\n{content}')
    return '\n\n'.join(result)


def make_rag_chain(question: str, retriever, model):
    """
    [변경] normalize_question으로 영어 쿼리 추출 → mmr_retrieve 사용
    retriever 인자는 호환성 유지를 위해 그대로 받되, 내부적으로 store 직접 접근
    """
    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'This system is designed for OCONUS beneficiaries, '
        'primarily Korean residents and USFK (주한미군) personnel.\n'
        'IMPORTANT OCONUS RULES:\n'
        '- In South Korea, TRICARE is the PRIMARY payer (not Medicare).\n'
        '- Medicare does NOT cover overseas medical expenses.\n'
        '- Overseas claims require pay-up-front then submit within 3 years.\n'
        '- Medicare Part B must be actively enrolled (not automatic) for overseas residents.\n\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n'
        '보험 추천이나 특정 플랜을 권유하는 답변은 하지 마세요. '
        '정보 제공만 하고 최종 판단은 사용자 본인과 보험사에게 맡기세요.\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- The answer language must match the user\'s question language.\n'
        f'- Do not switch languages unless necessary for official terms. '
        f'Use term locking format: 본인부담금(Copay), 공제액(Deductible), 사전승인(Prior Authorization).\n\n'
        '[참고 문서]\n{context}\n\n'
        '[질문]\n{question}\n\n'
        '[답변]\n'
    )

    rag_prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

    return (
        {'question': RunnablePassthrough(), 'context': retriever | format_docs}
        | rag_prompt
        | model
        | StrOutputParser()
    )


# mmr_retrieve를 활용한 make_rag_chain_v2 — normalize_question 완전 통합 버전
# Demo에서는 make_rag_chain(기존 호환) 또는 make_rag_chain_v2 중 선택 가능
def make_rag_chain_v2(question: str, store, model):
    """normalize_question + mmr_retrieve 완전 통합 버전"""
    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')
    norm   = normalize_question(question)
    docs   = mmr_retrieve(question, store)
    context = format_docs(docs)

    PROMPT_TEMPLATE = (
        'You are a TRICARE health benefits specialist.\n'
        'This system is designed for OCONUS beneficiaries, '
        'primarily Korean residents and USFK (주한미군) personnel.\n'
        'IMPORTANT OCONUS RULES:\n'
        '- In South Korea, TRICARE is the PRIMARY payer (not Medicare).\n'
        '- Medicare does NOT cover overseas medical expenses.\n'
        '- Overseas claims require pay-up-front then submit within 3 years.\n'
        '- Medicare Part B must be actively enrolled (not automatic) for overseas residents.\n\n'
        f'Query intent: {norm.get("intent", "general")} | Region: {norm.get("region", "unknown")}\n\n'
        'Answer questions based ONLY on the provided TRICARE documents below.\n'
        'If the answer is not in the documents, '
        'say "해당 내용은 제공된 문서에서 확인되지 않습니다." in the user\'s language.\n'
        'Always mention relevant conditions such as Group A/B, plan type, or beneficiary status.\n'
        '보험 추천이나 특정 플랜을 권유하는 답변은 하지 마세요.\n\n'
        f'IMPORTANT LANGUAGE RULE:\n'
        f'- You must answer in {answer_language}.\n'
        f'- Use term locking: 본인부담금(Copay), 공제액(Deductible), 사전승인(Prior Authorization).\n\n'
        f'[참고 문서]\n{context}\n\n'
        f'[질문]\n{question}\n\n'
        '[답변]\n'
    )

    from langchain_core.messages import HumanMessage
    resp = model.invoke([HumanMessage(content=PROMPT_TEMPLATE)])
    return resp.content, docs


print('✅ LLM + RAG 체인 구성 완료 (v2 — normalize_question + MMR 통합)')
print('   make_rag_chain(q, retriever, model)  → 기존 호환 버전')
print('   make_rag_chain_v2(q, store, model)   → normalize_question + MMR 완전 통합 버전')


✅ LLM + RAG 체인 구성 완료 (v2 — normalize_question + MMR 통합)
   make_rag_chain(q, retriever, model)  → 기존 호환 버전
   make_rag_chain_v2(q, store, model)   → normalize_question + MMR 완전 통합 버전


---
# 🔬 실험 섹션
## Section EX-1 · BM25Retriever 단독

### BM25란?
- 키워드 빈도(TF)와 문서 길이를 고려하는 통계 기반 검색
- 벡터 임베딩 없이 **단어 그 자체**로 매칭
- 'Group A', 'copay', 'deductible' 같은 정확한 용어 검색에 강함
- 반대로 의미가 비슷한 다른 단어('비용'→'cost')는 못 잡음


In [17]:
#!pip install rank-bm25  # 처음 실행 시 주석 해제

In [18]:
from langchain_community.retrievers import BM25Retriever

# BM25는 벡터 DB가 아닌 Document 리스트 직접 사용
# all_text_chunks는 Section 7에서 만들어진 통합 청크 리스트
bm25_retriever = BM25Retriever.from_documents(
    all_text_chunks,
    k=6  # 반환할 문서 수 (MMR과 동일하게 맞춤)
)

# BM25 단독 테스트
test_q = 'TRICARE Prime 한국 overseas copay Group A'
bm25_results = bm25_retriever.invoke(test_q)

print(f'🔍 BM25 검색 결과: {len(bm25_results)}개')
for i, doc in enumerate(bm25_results):
    src  = doc.metadata.get('source_file', doc.metadata.get('source', '?'))
    loc  = doc.metadata.get('location', '?')
    page = doc.metadata.get('page', '')
    print(f'  [{i+1}] {src} | loc={loc} | p.{page}')
    print(f'       {doc.page_content[:80].replace(chr(10), " ")}...')


🔍 BM25 검색 결과: 6개
  [1] TricarePlans.csv | loc=BOTH | p.
       항목: What is it? 플랜명: TRICARE Prime Remote Overseas 내용: A managed care option off...
  [2] Overseas_HB(해외 프로그램 안내서).pdf | loc=OCONUS | p.4
       HEALTH CARE OPTIONS  TRICARE Prime Overseas Options  TRICARE Prime Overseas and ...
  [3] Costs_Fees.pdf | loc=BOTH | p.0
       TRICARE® 2026 Costs and Fees TRICARE For Life, survivors, and medically retired ...
  [4] Overseas_HB(해외 프로그램 안내서).pdf | loc=OCONUS | p.8
       TRICARE For Life  TRICARE For Life is Medicare-wraparound coverage for TRICARE b...
  [5] Retiring_NGR_Br.pdf | loc=BOTH | p.1
       WHAT TRICARE PROGRAM OPTIONS DO I HAVE? For cost details, visit www.tricare.mil/...
  [6] Pharmacy_HB(tricare 약국 프로그램 안내서).pdf | loc=BOTH | p.19
       Costs by Beneficiary Group  Cost-shares, deductibles, and catastrophic caps are ...


---
## Section EX-2 · Hybrid Search (BM25 + MMR 결합)

### Hybrid Search란?
- BM25(키워드 검색) 결과 + 벡터 MMR 결과를 **가중치로 합산**
- `alpha` 값으로 두 검색의 비율 조절
  - `alpha=0.0` → BM25 100%
  - `alpha=1.0` → 벡터 검색 100%
  - `alpha=0.5` → 절반씩 (기본값)
- 키워드가 정확한 경우도, 의미만 비슷한 경우도 모두 잡을 수 있음


In [ ]:
#!pip install langchain-community rank-bm25
#!pip install -U langchain

1. 함수 정의 부분

In [21]:
# EnsembleRetriever 대신 직접 구현
# BM25 결과 + 벡터 결과를 순위 기반으로 합산 (RRF 방식)

def hybrid_retrieve(question: str, bm25_ret, vec_ret,
                    bm25_weight: float = 0.4, vec_weight: float = 0.6,
                    k: int = 6) -> list:
    """
    두 retriever 결과를 가중치로 합산해서 상위 k개 반환
    bm25_weight + vec_weight = 1.0 이 되도록 설정
    """
    bm25_docs = bm25_ret.invoke(question)   # BM25 검색 결과
    vec_docs  = vec_ret.invoke(question)    # 벡터(MMR) 검색 결과

    scores  = {}   # 문서별 합산 점수
    doc_map = {}   # 문서 저장용

    # BM25 결과에 가중치 적용
    # 1위 → bm25_weight * 1/1, 2위 → bm25_weight * 1/2, ...
    for rank, doc in enumerate(bm25_docs):
        key = doc.page_content[:100]  # 앞 100자를 고유 키로 사용
        scores[key]  = scores.get(key, 0) + bm25_weight * (1 / (rank + 1))
        doc_map[key] = doc

    # 벡터 결과에 가중치 적용 (같은 문서면 점수 누적)
    for rank, doc in enumerate(vec_docs):
        key = doc.page_content[:100]
        scores[key]  = scores.get(key, 0) + vec_weight * (1 / (rank + 1))
        doc_map[key] = doc

    # 점수 높은 순으로 정렬 후 상위 k개 반환
    sorted_keys = sorted(scores, key=lambda x: scores[x], reverse=True)
    return [doc_map[key] for key in sorted_keys[:k]]

In [22]:
#  Hybrid 단독 테스트 
# test_q는 셀 31에서 정의된 테스트 질문
hybrid_results = hybrid_retrieve(test_q, bm25_retriever, retriever)

print(f'🔍 Hybrid 검색 결과: {len(hybrid_results)}개')
for i, doc in enumerate(hybrid_results):
    src  = doc.metadata.get('source_file', doc.metadata.get('source', '?'))
    loc  = doc.metadata.get('location', '?')
    page = doc.metadata.get('page', '')
    print(f'  [{i+1}] {src} | loc={loc} | p.{page}')
    print(f'       {doc.page_content[:80].replace(chr(10), " ")}...')

🔍 Hybrid 검색 결과: 6개
  [1] TricarePlans.csv | loc=BOTH | p.
       항목: Where is the program available? 플랜명: TRICARE Prime Remote Overseas 내용: In de...
  [2] TricarePlans.csv | loc=BOTH | p.
       항목: What is it? 플랜명: TRICARE Prime Remote Overseas 내용: A managed care option off...
  [3] TricarePlans.csv | loc=BOTH | p.
       항목: What is the maximum I'll pay out-of-pocket? 플랜명: TRICARE Prime Overseas 내용: ...
  [4] Costs_Fees.pdf | loc=BOTH | p.0
       TRICARE® 2026 Costs and Fees TRICARE For Life, survivors, and medically retired ...
  [5] Overseas_HB(해외 프로그램 안내서).pdf | loc=OCONUS | p.4
       HEALTH CARE OPTIONS  TRICARE Prime Overseas Options  TRICARE Prime Overseas and ...
  [6] TricarePlans.csv | loc=BOTH | p.
       항목: How can I learn more? 플랜명: TRICARE Prime Overseas 내용: Go to the TRICARE Prim...


2. alpha 값 실험

In [ ]:
# alpha 값 실험 
# alpha = 벡터 비중
# alpha 0.3 → BM25 70% + 벡터 30% (키워드 매칭 강조)
# alpha 0.5 → BM25 50% + 벡터 50% (균등)
# alpha 0.7 → BM25 30% + 벡터 70% (의미 검색 강조)
# 출처 파일이 어떻게 달라지는지 확인하는 용도

print('\n--- alpha 값별 결과 출처 비교 ---')
for alpha in [0.3, 0.5, 0.7]:
    bm25_w = round(1 - alpha, 1)
    vec_w  = round(alpha, 1)
    results = hybrid_retrieve(test_q, bm25_retriever, retriever,
                              bm25_weight=bm25_w, vec_weight=vec_w)
    srcs = [d.metadata.get('source_file', '?') for d in results]
    print(f'  BM25={bm25_w} / 벡터={vec_w}: {srcs}')


--- alpha 값별 결과 출처 비교 ---
  BM25=0.7 / 벡터=0.3: ['?', 'Overseas_HB(해외 프로그램 안내서).pdf', '?', 'Costs_Fees.pdf', 'Overseas_HB(해외 프로그램 안내서).pdf', '?']
  BM25=0.5 / 벡터=0.5: ['?', '?', 'Costs_Fees.pdf', 'Overseas_HB(해외 프로그램 안내서).pdf', '?', '?']
  BM25=0.3 / 벡터=0.7: ['?', '?', '?', 'Costs_Fees.pdf', '?', 'Overseas_HB(해외 프로그램 안내서).pdf']


---
## Section EX-3 · Reranker (CrossEncoder)

### Rerank
- 1차 검색(BM25/벡터)으로 후보 문서를 뽑은 후
  **CrossEncoder 모델**로 질문-문서 쌍의 관련도를 다시 채점해 순서 조정
- CrossEncoder는 질문과 문서를 동시에 입력받아 정밀한 점수를 매김
- 1차 검색보다 훨씬 정확하지만 속도가 느림 → 후보 20개 뽑고 상위 6개만 남기는 방식

### 사용 모델
- `cross-encoder/ms-marco-MiniLM-L-6-v2`: 경량 영어 Reranker (빠름)
- `BAAI/bge-reranker-v2-m3`: 다국어 지원 Reranker (한국어 질문에 더 유리)


In [ ]:
#!pip install sentence-transformers

In [24]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')
# reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')


def rerank_docs(question: str, docs: list, top_k: int = 6) -> list:
    """
    CrossEncoder로 검색된 docs를 재점수 후 상위 top_k개 반환
    """
    if not docs:
        return docs
    pairs = []
    for doc in docs:
        content = doc.page_content
        if '[search_tags]' in content:
            content = content.split('[search_tags]')[0].strip()
        pairs.append((question, content))

    scores = reranker.predict(pairs)
    scored = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:top_k]]


# Hybrid 20개 후보 — EnsembleRetriever 없이 직접 구현
def hybrid_retrieve_wide(question: str, k: int = 20) -> list:
    """Rerank용 넓은 후보 수집 (k=20)"""
    bm25_wide = BM25Retriever.from_documents(all_text_chunks, k=k)
    vec_wide  = vector_store.as_retriever(
        search_type='mmr',
        search_kwargs={'k': k, 'fetch_k': 40}
    )
    return hybrid_retrieve(question, bm25_wide, vec_wide, k=k)


# Rerank 테스트
candidates = hybrid_retrieve_wide(test_q)
reranked_results = rerank_docs(test_q, candidates, top_k=6)

print(f'🔍 Rerank 결과: {len(candidates)}개 후보 → {len(reranked_results)}개 선별')
for i, doc in enumerate(reranked_results):
    src  = doc.metadata.get('source_file', doc.metadata.get('source', '?'))
    loc  = doc.metadata.get('location', '?')
    page = doc.metadata.get('page', '')
    print(f'  [{i+1}] {src} | loc={loc} | p.{page}')
    print(f'       {doc.page_content[:80].replace(chr(10), " ")}...')

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

c:\Users\kjw70\miniconda3\envs\nlp_env\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kjw70\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

🔍 Rerank 결과: 20개 후보 → 6개 선별
  [1] Costs_Fees.pdf | loc=BOTH | p.0
       TRICARE® 2026 Costs and Fees TRICARE For Life, survivors, and medically retired ...
  [2] Pharmacy_HB(tricare 약국 프로그램 안내서).pdf | loc=BOTH | p.19
       Costs by Beneficiary Group  Cost-shares, deductibles, and catastrophic caps are ...
  [3] Overseas_HB(해외 프로그램 안내서).pdf | loc=OCONUS | p.3
       YOUR TRICARE OPTIONS BY SPONSOR STATUS   Your TRICARE health plan options can ch...
  [4] Overseas_HB(해외 프로그램 안내서).pdf | loc=OCONUS | p.12
       TRICARE PHARMACY PROGRAM  The TRICARE Pharmacy Program provides prescription dru...
  [5] TricarePlans.csv | loc=BOTH | p.
       항목: What is it? 플랜명: TRICARE Prime Overseas 내용: A managed care option offering t...
  [6] TricarePlans.csv | loc=BOTH | p.
       항목: What is it? 플랜명: TRICARE Prime Remote Overseas 내용: A managed care option off...


---
## Section EX-4 · 4가지 방식 답변 비교

### 비교 대상
| 방식 | 검색 방법 | 특징 |
|---|---|---|
| A. MMR (기존) | 벡터 유사도만 | integrated5 현재 방식 |
| B. BM25 | 키워드 빈도만 | 정확한 단어 매칭 |
| C. Hybrid | BM25 + MMR 결합 | 두 방식 장점 결합 |
| D. Hybrid + Rerank | Hybrid 후 CrossEncoder 재정렬 | 가장 정밀한 방식 |

### 정성 평가 기준 (사람이 보고 판단)
- 답변에 Group A/B 구분이 명확히 나오는가?
- 출처 파일명이 질문과 관련된 파일인가?
- retrieved_docs가 다양한 파일에서 나오는가 (중복 없는가)?
- 답변이 문서에 없는 내용을 만들어내지 않는가 (hallucination)?


In [29]:
# 출처 표시 변경
# 원래는 source_file 대신 source라는 키를 사용함
# 출력 코드에서 sourch_file 못 찾으면 source 보도록 설정했으나 전체 경로에 들어 있어서 ?가 뜸


# 함수 추가
#import os
#
#def get_source_name(doc) -> str:
#    """
#    metadata에서 파일명만 추출
#    - source_file 있으면 그대로 사용 (PDF 청크)
#    - source만 있으면 경로에서 파일명만 추출 (CSV 청크)
#    """
#    if doc.metadata.get('source_file'):
#        return doc.metadata['source_file']
#    source = doc.metadata.get('source', '알 수 없음')
#    return os.path.basename(str(source))  # 경로에서 파일명만 추출
#  csv 파일은 직접 크롤링해서 만든 데이터 파일이기에 출처 표시 추가로 수정하기로 함

import os

# CSV 파일별 원본 출처 URL 매핑
# 각 CSV는 원아씨가 직접 만든 파일이지만 내용의 원본은 아래 공식 사이트
CSV_SOURCE_URLS = {
    'mental_health_services.csv':
        'https://www.tricare.mil/CoveredServices/Mental (tricare.mil 공식)',
    'TricarePlans.csv':
        'https://www.tricare.mil/Plans (tricare.mil 공식)',
    'tricare_exclusions.csv':
        'https://www.tricare.mil/CoveredServices/IsItCovered/Exclusions (tricare.mil 공식)',
    'Health_Plan_Costs.csv':
        'https://www.tricare.mil/Costs/Compare (tricare.mil 공식)',
}

def get_source_name(doc) -> str:
    """
    metadata에서 출처 표시용 이름 반환
    - PDF 청크: 파일명 그대로 (공식 PDF 문서)
    - CSV 청크: 파일명 대신 원본 공식 사이트 URL 표시
    """
    # PDF 청크: source_file 있으면 그대로 사용
    if doc.metadata.get('source_file'):
        return doc.metadata['source_file']

    # CSV 청크: source에서 파일명 추출 후 URL로 교체
    source = doc.metadata.get('source', '')
    fname  = os.path.basename(str(source))  # 경로에서 파일명만 추출

    # CSV 파일이면 공식 사이트 URL 반환
    if fname in CSV_SOURCE_URLS:
        return CSV_SOURCE_URLS[fname]

    return fname  # 매핑 없으면 파일명 그대로

In [34]:
from langchain_core.messages import HumanMessage

#  비교용 질문 세트 
# 각 질문은 특정 파일·특정 조건을 타겟으로 설계
eval_questions = [
    '한국에서 TRICARE Prime 외래 진료 본인부담금(Copay)은 얼마인가요? Group A 기준으로 알려주세요.',
    '해외에서 약을 구매했을 때 TRICARE로 환급받는 방법은 무엇인가요?',
    'TRICARE ADDP에서 OCONUS에 있는 군인이 치과 치료를 받으려면 어떻게 해야 하나요?',
    '예비군이 30일 이상 활성화될 때 가족의 TRICARE 플랜은 어떻게 변경되나요?',
]


def get_answer_from_docs(question: str, docs: list) -> str:
    """검색된 docs와 질문으로 LLM 답변 생성"""
    context = format_docs(docs)
    language_code   = detect_language(question)
    answer_language = LANGUAGE_NAME_MAP.get(language_code, 'English')

    prompt = (
        'You are a TRICARE health benefits specialist for OCONUS/Korea beneficiaries.\n'
        'Answer ONLY based on the documents below. '
        'If not in documents, say "해당 내용은 제공된 문서에서 확인되지 않습니다."\n'
        'Use term locking: 본인부담금(Copay), 공제액(Deductible), 사전승인(Prior Authorization).\n'
        f'Answer in {answer_language}.\n\n'
        f'[참고 문서]\n{context}\n\n'
        f'[질문]\n{question}\n\n[답변]\n'
    )
    resp = model.invoke([HumanMessage(content=prompt)])
    return resp.content


#  4가지 방식 비교 실행 
# 답변 출력 수정:print(f'  답변: {ans_rerank[:200]}.' ->  print(f'  답변: {ans_rerank}' = 전체 답변 보이도록 함 ([:200]}.) 제거
for q in eval_questions:
    print('\n' + '=' * 100)
    print(f'QUESTION: {q}')
    print('=' * 100)

    # A. MMR (기존 integrated5 방식)
    docs_mmr = retriever.invoke(q)
    ans_mmr  = get_answer_from_docs(q, docs_mmr)
    print(f'\n[A] MMR ({len(docs_mmr)}개 검색)')
    #print(f'  출처: {[d.metadata.get("source_file","?") for d in docs_mmr]}')
    print(f'  출처: {[get_source_name(d) for d in docs_mmr]}')
    print(f'  답변: {ans_mmr}')

    # B. BM25 단독
    docs_bm25 = bm25_retriever.invoke(q)
    ans_bm25  = get_answer_from_docs(q, docs_bm25)
    print(f'\n[B] BM25 ({len(docs_bm25)}개 검색)')
    #print(f'  출처: {[d.metadata.get("source_file","?") for d in docs_bm25]}')
    print(f'  출처: {[get_source_name(d) for d in docs_bm25]}')
    print(f'  답변: {ans_bm25}')

    # C. Hybrid
    docs_hybrid = hybrid_retrieve(q, bm25_retriever, retriever)
    ans_hybrid  = get_answer_from_docs(q, docs_hybrid)
    print(f'\n[C] Hybrid ({len(docs_hybrid)}개 검색)')
    #print(f'  출처: {[d.metadata.get("source_file","?") for d in docs_hybrid]}')
    print(f'  출처: {[get_source_name(d) for d in docs_hybrid]}')
    print(f'  답변: {ans_hybrid}')

    # D. Hybrid + Rerank
    candidates_d = hybrid_retrieve_wide(q)          
    docs_rerank  = rerank_docs(q, candidates_d, top_k=6)
    ans_rerank   = get_answer_from_docs(q, docs_rerank)
    print(f'\n[D] Hybrid+Rerank ({len(candidates_d)}개 후보→{len(docs_rerank)}개 선별)')
    print(f'  출처: {[get_source_name(d) for d in docs_rerank]}')
    #print(f'  출처: {[d.metadata.get("source_file","?") for d in docs_rerank]}')
    print(f'  답변: {ans_rerank}')



QUESTION: 한국에서 TRICARE Prime 외래 진료 본인부담금(Copay)은 얼마인가요? Group A 기준으로 알려주세요.

[A] MMR (6개 검색)
  출처: ['https://www.tricare.mil/Plans (tricare.mil 공식)', 'Costs_Fees.pdf', 'Overseas_HB(해외 프로그램 안내서).pdf', 'Costs_Fees.pdf', 'Costs_Fees.pdf', 'Overseas_HB(해외 프로그램 안내서).pdf']
  답변: 한국에서 TRICARE Prime 외래 진료 본인부담금(Copay)은 Group A 기준으로 $26입니다.

[B] BM25 (6개 검색)
  출처: ['Costs_Fees.pdf', 'Overseas_HB(해외 프로그램 안내서).pdf', 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf', 'Overseas_HB(해외 프로그램 안내서).pdf', 'Costs_Fees.pdf', 'Retiring_NGR_Br.pdf']
  답변: 한국에서 TRICARE Prime 외래 진료 본인부담금(Copay)은 Group A 기준으로 $26입니다.

[C] Hybrid (6개 검색)
  출처: ['Costs_Fees.pdf', 'https://www.tricare.mil/Plans (tricare.mil 공식)', 'Costs_Fees.pdf', 'Overseas_HB(해외 프로그램 안내서).pdf', 'Overseas_HB(해외 프로그램 안내서).pdf', 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf']
  답변: 한국에서 TRICARE Prime 외래 진료 본인부담금(Copay)은 Group A 기준으로 $26입니다.

[D] Hybrid+Rerank (20개 후보→6개 선별)
  출처: ['Costs_Fees.pdf', 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf', 'Costs_Fees.pdf', 'Overseas_H

---
## Section EX-5 · 정량 평가 — Hit Rate / MRR / Faithfulness

### 평가 지표 설명

| 지표 | 설명 | 공식 |
|---|---|---|
| **Hit Rate** | 정답 문서가 검색 결과 k개 안에 있는 비율 | 정답 포함 질문 수 / 전체 질문 수 |
| **MRR** | 정답 문서의 평균 순위 역수 | Σ(1/정답_순위) / 전체 질문 수 |
| **Faithfulness** | 답변이 문서에 근거하는 비율 | LLM-as-judge로 0~1 점수 |

### 평가 데이터셋 구성
- 각 질문마다 **정답 파일명**을 미리 지정해둠
- 검색 결과에 그 파일이 포함되면 Hit, 포함 안 되면 Miss

### Faithfulness 판단 방법 (LLM-as-judge)
- 답변의 각 문장이 검색된 문서에 근거하는지 LLM에게 물어봄
- RAGAS 라이브러리를 쓰거나 직접 프롬프트로 구현 가능


In [36]:
# 평가 데이터셋 
# (질문, 정답_파일명, 정답_키워드) 형태
# 정답_파일명: 이 파일의 청크가 검색 결과에 있으면 Hit
# 정답_키워드: 답변에 이 단어가 있으면 답변 품질 OK로 판단
EVAL_DATASET = [
    {
        'question':    '한국에서 TRICARE Prime Overseas 본인부담금(Copay)은 얼마인가요?',
        'answer_file': 'Costs_Fees.pdf',
        'answer_kw':   ['$0', 'no cost', 'overseas'],
    },
    {
        'question':    '해외 약국에서 처방약 구매 후 환급 신청 방법은?',
        'answer_file': 'Pharmacy_HB(tricare 약국 프로그램 안내서).pdf',
        'answer_kw':   ['claim', '환급', 'overseas'],
    },
    {
        'question':    'TRICARE ADDP OCONUS에서 응급 치과 치료 받는 방법은?',
        'answer_file': 'ADDP_Brochure_FINAL_122624_508c.pdf',
        'answer_kw':   ['emergency', 'ACN', 'OCONUS'],
    },
    {
        'question':    '예비군 30일 이상 활성화 시 가족 TRICARE 플랜 변경은?',
        'answer_file': 'NGR_HB(국가방위군 및 예비군을 위한 트라이케어 안내서).pdf',
        'answer_kw':   ['TRICARE Prime', 'active duty', '활성화'],
    },
    {
        'question':    'QLE로 인정되는 생애사건에는 어떤 것들이 있나요?',
        'answer_file': 'QLEs_FS (2).pdf',
        'answer_kw':   ['marriage', '결혼', 'birth', '출생'],
    },
]


def calc_hit_rate_mrr(results_per_q: list, eval_dataset: list) -> dict:
    """
    Hit Rate와 MRR 계산
    results_per_q: 각 질문에 대한 검색 결과 docs 리스트의 리스트
    """
    hits, mrr_sum = 0, 0.0
    for item, docs in zip(eval_dataset, results_per_q):
        target_file = item['answer_file']
        for rank, doc in enumerate(docs, start=1):
            sf = doc.metadata.get('source_file', doc.metadata.get('source', ''))
            if target_file in sf:
                hits    += 1
                mrr_sum += 1 / rank
                break
    n = len(eval_dataset)
    return {'hit_rate': hits / n, 'mrr': mrr_sum / n, 'hits': hits, 'total': n}


def faithfulness_judge(question: str, answer: str, docs: list) -> float:
    """
    LLM-as-judge: 답변이 문서에 근거하는지 0~1 점수로 평가
    """
    context = format_docs(docs)
    judge_prompt = (
        'You are an objective evaluator.\n'
        'Given the retrieved documents and an answer, '
        'rate how faithfully the answer is grounded in the documents.\n'
        'Score: 0.0 (completely hallucinated) to 1.0 (fully grounded).\n'
        'Respond ONLY with a number between 0.0 and 1.0.\n\n'
        f'[Documents]\n{context[:1500]}\n\n'
        f'[Question]\n{question}\n\n'
        f'[Answer]\n{answer}\n\n'
        'Score:'
    )
    resp = model.invoke([HumanMessage(content=judge_prompt)])
    try:
        return float(resp.content.strip())
    except:
        return 0.5


#  4가지 방식 정량 평가 실행 
methods = {
    'A_MMR':lambda q: retriever.invoke(q),
    'B_BM25':lambda q: bm25_retriever.invoke(q),
    'C_Hybrid':lambda q: hybrid_retrieve(q, bm25_retriever, retriever),
    'D_Hybrid+Rerank': lambda q: rerank_docs(q, hybrid_retrieve_wide(q), top_k=6),
} 

print('\n' + '='*60)
print('📊 정량 평가 결과')
print('='*60)

for method_name, retrieve_fn in methods.items():
    all_results   = []
    faithfulness_scores = []

    for item in EVAL_DATASET:
        q    = item['question']
        docs = retrieve_fn(q)
        all_results.append(docs)

        # Faithfulness 평가 (비용 절약을 위해 A, D만 실행해도 됨)
        ans    = get_answer_from_docs(q, docs)
        faith  = faithfulness_judge(q, ans, docs)
        faithfulness_scores.append(faith)

    metrics = calc_hit_rate_mrr(all_results, EVAL_DATASET)
    avg_faith = sum(faithfulness_scores) / len(faithfulness_scores)

    print(f'\n[{method_name}]')
    print(f'  Hit Rate : {metrics["hit_rate"]:.2f}  ({metrics["hits"]}/{metrics["total"]})')
    print(f'  MRR : {metrics["mrr"]:.3f}')
    print(f'  Faithfulness: {avg_faith:.3f}')

print('\n✅ 평가 완료. Hit Rate·MRR이 높고 Faithfulness가 높은 방식이 최적')



📊 정량 평가 결과

[A_MMR]
  Hit Rate : 1.00  (5/5)
  MRR : 0.667
  Faithfulness: 0.920

[B_BM25]
  Hit Rate : 0.40  (2/5)
  MRR : 0.250
  Faithfulness: 0.900

[C_Hybrid]
  Hit Rate : 1.00  (5/5)
  MRR : 0.573
  Faithfulness: 1.000

[D_Hybrid+Rerank]
  Hit Rate : 1.00  (5/5)
  MRR : 0.867
  Faithfulness: 1.000

✅ 평가 완료. Hit Rate·MRR이 높고 Faithfulness가 높은 방식이 최적


---
## Section EX-6 · 결과 해석 

### 지표 해석

```
Hit Rate
  1.0 = 모든 질문에서 정답 파일 포함 (완벽)
  0.8 = 5개 중 4개 질문에서 정답 파일 포함 (양호)
  0.6 이하 = 개선 필요

MRR
  1.0 = 모든 질문에서 정답 파일이 1위로 검색됨
  0.5 = 평균 2위 수준
  0.3 이하 = 정답이 뒤쪽에 밀려있어 LLM 컨텍스트 품질 낮음

Faithfulness
  0.9 이상 = 답변이 문서에 거의 완전히 근거함
  0.7~0.9  = 대부분 근거하나 일부 추론 포함
  0.7 미만  = hallucination 위험 있음
```


---

# 중간 점검 1 (유사도 검사)

In [37]:
#  유사도 점수 직접 확인 
# Chroma는 거리(distance) 기반 → 숫자가 낮을수록 질문과 더 유사
# 0에 가까울수록 좋음

test_questions = [
    '한국에서 TRICARE Prime 본인부담금',
    '해외 약국 환급 신청',
    'OCONUS 응급 치과 치료',
]

for q in test_questions:
    print(f'\n질문: {q}')
    print('-' * 60)

    results = vector_store.similarity_search_with_score(q, k=6)

    for doc, score in results:
        src  = doc.metadata.get('source_file', doc.metadata.get('source', '?'))
        loc  = doc.metadata.get('location', '?')
        page = doc.metadata.get('page', '')
        print(f'  거리: {score:.4f} | {src} | loc={loc} | p.{page}')
        print(f'  내용: {doc.page_content[:80].replace(chr(10), " ")}')


질문: 한국에서 TRICARE Prime 본인부담금
------------------------------------------------------------
  거리: 0.6552 | Costs_Fees.pdf | loc=BOTH | p.0
  내용: TRICARE® 2026 Costs and Fees TRICARE For Life, survivors, and medically retired 
  거리: 0.6584 | Overseas_HB(해외 프로그램 안내서).pdf | loc=OCONUS | p.5
  내용: HEALTH CARE OPTIONS  TRICARE Prime Overseas Options (continued)  TRICARE PRIME O
  거리: 0.6776 | TricarePlans.csv | loc=BOTH | p.
  내용: 항목: Will I have to file my own claims? 플랜명: TRICARE Prime 내용: Your provider will
  거리: 0.6908 | TricarePlans.csv | loc=BOTH | p.
  내용: 항목: Where is the program available? 플랜명: TRICARE Prime 내용: In the United States 
  거리: 0.6913 | TricarePlans.csv | loc=BOTH | p.
  내용: 항목: What is the maximum I'll pay out-of-pocket? 플랜명: TRICARE Prime 내용: To see co
  거리: 0.6976 | TricarePlans.csv | loc=BOTH | p.
  내용: 항목: Will I have to file my own claims? 플랜명: TRICARE Prime Overseas 내용: Your prov

질문: 해외 약국 환급 신청
------------------------------------------------------------
  거리: 0

# 중간 점검2 (한국어 쿼리 vs 영어 변환 쿼리 유사도 비교)

In [38]:
#  한국어 쿼리 vs 영어 변환 쿼리 유사도 비교 
# normalize_question()이 한국어를 영어로 변환해주는데
# 변환 전후 거리 차이를 보면 영어 변환이 실제로 도움이 되는지 확인 가능

compare_questions = [
    '해외 약국 환급 신청',
    'OCONUS 응급 치과 치료',
    '한국에서 TRICARE Prime 본인부담금',
]

for q_ko in compare_questions:
    # 영어 쿼리 추출
    norm   = normalize_question(q_ko)
    q_en   = norm['english_query']

    # 한국어로 검색
    res_ko = vector_store.similarity_search_with_score(q_ko, k=3)
    # 영어로 검색
    res_en = vector_store.similarity_search_with_score(q_en, k=3)

    print(f'\n질문 (한국어): {q_ko}')
    print(f'질문 (영어변환): {q_en}')
    print(f'{"─"*60}')

    # 한국어 결과
    print('  [한국어 검색]')
    for doc, score in res_ko:
        src = doc.metadata.get('source_file', doc.metadata.get('source', '?'))
        print(f'    거리: {score:.4f} | {src}')

    # 영어 결과
    print('  [영어 변환 검색]')
    for doc, score in res_en:
        src = doc.metadata.get('source_file', doc.metadata.get('source', '?'))
        print(f'    거리: {score:.4f} | {src}')

    # 1위 거리 차이
    ko_best = res_ko[0][1]
    en_best = res_en[0][1]
    diff    = ko_best - en_best
    print(f'\n  1위 거리 차이: {ko_best:.4f} → {en_best:.4f} ({"개선 ✅" if diff > 0 else "차이 없음 ➡️"}  {abs(diff):.4f})')


질문 (한국어): 해외 약국 환급 신청
질문 (영어변환): How do I apply for reimbursement for overseas pharmacy expenses?
────────────────────────────────────────────────────────────
  [한국어 검색]
    거리: 0.7518 | TFL_HB(평생 트라이케어).pdf
    거리: 0.8395 | Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
    거리: 0.8463 | Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
  [영어 변환 검색]
    거리: 0.6633 | TFL_HB(평생 트라이케어).pdf
    거리: 0.6926 | Pharmacy_HB(tricare 약국 프로그램 안내서).pdf
    거리: 0.7387 | Pharmacy_HB(tricare 약국 프로그램 안내서).pdf

  1위 거리 차이: 0.7518 → 0.6633 (개선 ✅  0.0885)

질문 (한국어): OCONUS 응급 치과 치료
질문 (영어변환): Emergency dental treatment overseas
────────────────────────────────────────────────────────────
  [한국어 검색]
    거리: 0.7873 | ADDP_Brochure_FINAL_122624_508c.pdf
    거리: 0.8042 | TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf
    거리: 0.8882 | TRICARE_ADDP_HB_FINAL_508c(현역 군인 치과 프로그램 안내서).pdf
  [영어 변환 검색]
    거리: 0.7654 | ADDP_Brochure_FINAL_122624_508c.pdf
    거리: 0.8230 | Overseas_HB(해외 프로그램 안내서).pdf
    거리: 0.8444 | TRICARE_ADDP_HB_FI